# 🚀 ACE-Net Baseline: Zero-Shot FakeAVCeleb (N=700) Evaluation
### **Model**: Stage 2 Baseline (Unadapted) (`best_baseline_model.pth` / `stage2_acenet.pt`)
### **Benchmark**: Aligned 700 FakeAVCeleb clips (350 Real / 350 Fake)
### **Outputs**: Saved directly to Google Drive `Baseline_training/checkpoints/preds_acenet_zeroshot_700.csv`

---
> **Note:** This notebook evaluates the **Zero-Shot** capability of ACE-Net. It tests how well the model discriminates deepfakes using only emotion inconsistency representations learned from MELD and CREMA-D without any prior adaptation or exposure to FakeAVCeleb.

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import torch
from pathlib import Path

drive.mount('/content/drive')

print('GPU Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name   :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not active! Go to Runtime -> Change runtime type -> T4 GPU.')

## Step 2: Clone Baseline Repository & Install Dependencies

In [ ]:
import os
%cd /content
if not os.path.exists('/content/Baseline_Training'):
    !git clone https://github.com/gjvlio/Baseline_Training.git
%cd /content/Baseline_Training
!git checkout feat/acenet-whisper-eval-700
!git pull

# Install all required evaluation dependencies
!pip install -q scikit-learn transformers librosa openai-whisper facenet-pytorch
print('✅ Environment ready!')

## Step 3: Verify Drive Paths & Zero-Shot Checkpoint

In [ ]:
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/Baseline_training')
CKPT_DIR = DRIVE_BASE / 'checkpoints'

# Check for zero-shot checkpoint
ZERO_SHOT_CKPT = CKPT_DIR / 'best_baseline_model.pth'
if not ZERO_SHOT_CKPT.exists():
    ZERO_SHOT_CKPT = CKPT_DIR / 'stage2_acenet.pt'

EVAL_MANIFEST = Path('Manifests/fakeavceleb_eval_700.csv')
ZIP_PATH = DRIVE_BASE / 'Copy of fakeavceleb.zip'
if not ZIP_PATH.exists():
    ZIP_PATH = DRIVE_BASE / 'fakeavceleb.zip'

OUTPUT_CSV = CKPT_DIR / 'preds_acenet_zeroshot_700.csv'

print('=' * 75)
print('📁 CONFIGURATION OVERVIEW:')
print(f'  Zero-Shot Checkpoint : {ZERO_SHOT_CKPT} (Exists: {ZERO_SHOT_CKPT.exists()})')
print(f'  Dataset Zip Archive  : {ZIP_PATH} (Exists: {ZIP_PATH.exists()})')
print(f'  Target Manifest (700): {EVAL_MANIFEST} (Exists: {EVAL_MANIFEST.exists()})')
print(f'  Target Output CSV    : {OUTPUT_CSV}')
print('=' * 75)

assert ZERO_SHOT_CKPT.exists(), f'❌ Zero-Shot Checkpoint not found at: {ZERO_SHOT_CKPT}'
assert ZIP_PATH.exists(), f'❌ Dataset zip not found at: {ZIP_PATH}'
assert EVAL_MANIFEST.exists(), f'❌ Manifest not found at: {EVAL_MANIFEST}'
print('✅ All input files verified successfully!')

## Step 4: Fast Selective Unzip of the 700 Evaluation Clips (~20 Seconds)
Extracts ONLY the 700 evaluation clips directly from the zip to Colab local disk.

In [ ]:
%cd /content/Baseline_Training

!python -m scripts.extract_700_videos \
    --zip-path '/content/drive/MyDrive/Baseline_training' \
    --manifest 'Manifests/fakeavceleb_eval_700.csv' \
    --output-dir '/content/fakeav_raw'

## Step 5: Run ZERO-SHOT Evaluation on FakeAVCeleb (N=700)
Evaluates `best_baseline_model.pth` directly on the 700 clips without fine-tuning.

In [ ]:
%cd /content/Baseline_Training

!python -u -m scripts.evaluate_fakeavceleb_700 \
    --manifest 'Manifests/fakeavceleb_eval_700.csv' \
    --raw-dir '/content/fakeav_raw' \
    --ckpt '/content/drive/MyDrive/Baseline_training/checkpoints/best_baseline_model.pth' \
    --output-csv '/content/drive/MyDrive/Baseline_training/checkpoints/preds_acenet_zeroshot_700.csv' \
    --cache-dir '/content/fakeav_cache_zeroshot_700' \
    --whisper-model base \
    --device cuda

## Step 6: Inspect Results and Metrics Saved in Google Drive

In [ ]:
import pandas as pd
from pathlib import Path

preds_file = Path('/content/drive/MyDrive/Baseline_training/checkpoints/preds_acenet_zeroshot_700.csv')
metrics_file = Path('/content/drive/MyDrive/Baseline_training/checkpoints/metrics_preds_acenet_zeroshot_700.csv')

if preds_file.exists():
    df_preds = pd.read_csv(preds_file)
    print(f'✅ Zero-shot predictions saved successfully! Total rows: {len(df_preds):,}')
    display(df_preds.head(10))

if metrics_file.exists():
    df_metrics = pd.read_csv(metrics_file)
    print('\n📊 ZERO-SHOT OFFICIAL METRICS:')
    display(df_metrics)